In [1]:
"""
Diagnostic Script - Rising Val Loss
Checks for common causes: vocab mismatch, data leakage, token distribution issues.
"""

import os
import torch
import torch.nn as nn
from collections import Counter

# ─────────────────────────────────────────────
# PATHS — adjust to match your setup
# ─────────────────────────────────────────────
TRAIN_PREPROCESSED_PATH = "/home/jovyan/lab3/coco_train_preprocessed.pt"
VAL_PREPROCESSED_PATH   = "/home/jovyan/lab3/coco_val_preprocessed_new.pt"
TRAIN_FEATURES_PATH     = "/home/jovyan/annabell/train_features.pt"
VAL_FEATURES_PATH       = "/home/jovyan/annabell/val_features.pt"

print("=" * 60)
print("DIAGNOSTIC: Rising Val Loss")
print("=" * 60)

# ─────────────────────────────────────────────
# 1. LOAD BOTH DATASETS
# ─────────────────────────────────────────────
print("\n[1] Loading datasets...")
train_data = torch.load(TRAIN_PREPROCESSED_PATH, weights_only=False)
val_data   = torch.load(VAL_PREPROCESSED_PATH,   weights_only=False)

train_samples = train_data["samples"]
val_samples   = val_data["samples"]
train_vocab   = train_data["vocab"]
val_vocab     = val_data["vocab"]

print(f"  Train samples : {len(train_samples)}")
print(f"  Val samples   : {len(val_samples)}")
print(f"  Train vocab size: {len(train_vocab)}")
print(f"  Val vocab size  : {len(val_vocab)}")

# ─────────────────────────────────────────────
# 2. VOCAB MISMATCH CHECK
# ─────────────────────────────────────────────
print("\n[2] Checking vocab mismatch...")

if train_vocab == val_vocab:
    print("  ✓ Vocabs are identical — no mismatch")
else:
    print("  ✗ VOCABS ARE DIFFERENT — this is likely your problem!")
    train_keys = set(train_vocab.keys())
    val_keys   = set(val_vocab.keys())
    only_train = train_keys - val_keys
    only_val   = val_keys - train_keys
    print(f"    Words only in train vocab: {len(only_train)}")
    print(f"    Words only in val vocab:   {len(only_val)}")
    print(f"    Example words only in val: {list(only_val)[:10]}")

# Check if special tokens have the same IDs
print("\n  Special token ID check:")
for token in ["<pad>", "<unk>", "<start>", "<end>"]:
    t_id = train_vocab.get(token, "MISSING")
    v_id = val_vocab.get(token, "MISSING")
    match = "✓" if t_id == v_id else "✗ MISMATCH"
    print(f"    {token}: train={t_id}, val={v_id} {match}")

# ─────────────────────────────────────────────
# 3. MAX LENGTH CHECK
# ─────────────────────────────────────────────
print("\n[3] Checking max_length...")
t_max = train_data["max_length"]
v_max = val_data["max_length"]
if t_max == v_max:
    print(f"  ✓ max_length matches: {t_max}")
else:
    print(f"  ✗ max_length MISMATCH: train={t_max}, val={v_max}")

# ─────────────────────────────────────────────
# 4. TOKEN ID DISTRIBUTION CHECK
# ─────────────────────────────────────────────
print("\n[4] Checking token ID distributions...")

def get_token_stats(samples):
    all_ids = []
    for s in samples[:1000]:  # sample first 1000 for speed
        all_ids.extend(s["token_ids"].tolist())
    return Counter(all_ids)

train_counter = get_token_stats(train_samples)
val_counter   = get_token_stats(val_samples)

train_unk_rate = train_counter.get(1, 0) / sum(train_counter.values())
val_unk_rate   = val_counter.get(1, 0)   / sum(val_counter.values())

print(f"  Train <unk> rate: {train_unk_rate:.4f} ({train_unk_rate*100:.2f}%)")
print(f"  Val   <unk> rate: {val_unk_rate:.4f} ({val_unk_rate*100:.2f}%)")

if val_unk_rate > train_unk_rate * 2:
    print("  ✗ Val has much higher <unk> rate — val was likely preprocessed with a different vocab!")
else:
    print("  ✓ <unk> rates look similar")

# ─────────────────────────────────────────────
# 5. FEATURE COVERAGE CHECK
# ─────────────────────────────────────────────
print("\n[5] Checking feature coverage...")
train_features = torch.load(TRAIN_FEATURES_PATH, weights_only=False)
val_features   = torch.load(VAL_FEATURES_PATH,   weights_only=False)

train_feat_keys = {os.path.basename(k) for k in train_features.keys()}
val_feat_keys   = {os.path.basename(k) for k in val_features.keys()}

train_sample_keys = {os.path.basename(s["image_path"]) for s in train_samples}
val_sample_keys   = {os.path.basename(s["image_path"]) for s in val_samples}

train_coverage = len(train_feat_keys & train_sample_keys) / len(train_sample_keys) * 100
val_coverage   = len(val_feat_keys   & val_sample_keys)   / len(val_sample_keys)   * 100

print(f"  Train feature coverage: {train_coverage:.1f}%")
print(f"  Val   feature coverage: {val_coverage:.1f}%")

if val_coverage < 90:
    print("  ✗ Val feature coverage is low — many val samples have no matching feature!")
else:
    print("  ✓ Feature coverage looks fine")

# Check for data leakage (val images appearing in train features)
overlap = train_feat_keys & val_feat_keys
if overlap:
    print(f"\n  ✗ WARNING: {len(overlap)} images appear in BOTH train and val features — data leakage!")
else:
    print(f"\n  ✓ No overlap between train and val image features")

# ─────────────────────────────────────────────
# 6. SAMPLE CAPTION SANITY CHECK
# ─────────────────────────────────────────────
print("\n[6] Sample captions sanity check...")
print("  First 3 train captions:")
for s in train_samples[:3]:
    print(f"    {s['caption']}")
    print(f"    token_ids: {s['token_ids'][:10].tolist()}...")

print("  First 3 val captions:")
for s in val_samples[:3]:
    print(f"    {s['caption']}")
    print(f"    token_ids: {s['token_ids'][:10].tolist()}...")

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
vocab_ok    = train_vocab == val_vocab
max_len_ok  = t_max == v_max
unk_ok      = val_unk_rate <= train_unk_rate * 2
coverage_ok = val_coverage >= 90
leakage_ok  = len(overlap) == 0

print(f"  Vocab match:        {'✓' if vocab_ok    else '✗ PROBLEM'}")
print(f"  max_length match:   {'✓' if max_len_ok  else '✗ PROBLEM'}")
print(f"  <unk> rate OK:      {'✓' if unk_ok      else '✗ PROBLEM'}")
print(f"  Feature coverage:   {'✓' if coverage_ok else '✗ PROBLEM'}")
print(f"  No data leakage:    {'✓' if leakage_ok  else '✗ PROBLEM'}")

if all([vocab_ok, max_len_ok, unk_ok, coverage_ok, leakage_ok]):
    print("\n  All checks passed — rising val loss is likely genuine overfitting.")
    print("  Suggested fixes: add dropout, reduce epochs, or lower learning rate.")
else:
    print("\n  One or more problems found — fix these before concluding overfitting.")

DIAGNOSTIC: Rising Val Loss

[1] Loading datasets...
  Train samples : 591753
  Val samples   : 25014
  Train vocab size: 10307
  Val vocab size  : 10307

[2] Checking vocab mismatch...
  ✓ Vocabs are identical — no mismatch

  Special token ID check:
    <pad>: train=0, val=0 ✓
    <unk>: train=1, val=1 ✓
    <start>: train=2, val=2 ✓
    <end>: train=3, val=3 ✓

[3] Checking max_length...
  ✓ max_length matches: 30

[4] Checking token ID distributions...
  Train <unk> rate: 0.0017 (0.17%)
  Val   <unk> rate: 0.0022 (0.22%)
  ✓ <unk> rates look similar

[5] Checking feature coverage...
  Train feature coverage: 100.0%
  Val   feature coverage: 100.0%
  ✓ Feature coverage looks fine

  ✓ No overlap between train and val image features

[6] Sample captions sanity check...
  First 3 train captions:
    Closeup of bins of food that include broccoli and bread.
    token_ids: [2, 4, 5, 6, 5, 7, 8, 9, 10, 11]...
    A meal is presented in brightly colored plastic trays.
    token_ids: [2, 13